In [36]:
%pip install --upgrade pip
%pip install "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [37]:
import pandas as pd
import numpy as np
import psycopg

In [38]:
conn = psycopg.connect(
    host="localhost",
    port='5432',
    dbname="postgres",
    user="postgres",
    password="postgres")

cur = conn.cursor()
print("Connected:", conn.info.dbname)

Connected: postgres


In [39]:
cur.execute("SELECT version();")
print(cur.fetchone())

('PostgreSQL 18.6 (Postgres.app) on aarch64-apple-darwin23.6.0, compiled by Apple clang version 15.0.0 (clang-1500.3.9.4), 64-bit',)


In [40]:
try:
    cur.execute("""
        DROP TABLE IF EXISTS company CASCADE;
        DROP TABLE IF EXISTS ceo CASCADE;
        DROP TABLE IF EXISTS city CASCADE;
        DROP TABLE IF EXISTS state CASCADE;
        DROP TABLE IF EXISTS country CASCADE;
    """)
    conn.commit()
    print("Dropped")
except Exception as e:
    conn.rollback()
    print("Error:", e)

Dropped


In [41]:
try:
    cur.execute("""
        CREATE TABLE city (
            city_id      SERIAL PRIMARY KEY,
            city_name    VARCHAR(100) NOT NULL,
            state_name   VARCHAR(100),            -- NULL when no state is listed
            country_name VARCHAR(100) NOT NULL
        );

        -- Prevents duplicate cities, even when state_name is NULL
        CREATE UNIQUE INDEX city_unique
            ON city (city_name, COALESCE(state_name, ''), country_name);
    """)
    conn.commit()
    print("city created")
except Exception as e:
    conn.rollback()
    print("Error:", e)

city created


In [42]:
try:
    cur.execute("""
        CREATE TABLE ceo (
            ceo_id         SERIAL PRIMARY KEY,
            first_name     VARCHAR(50) NOT NULL,
            last_name      VARCHAR(50) NOT NULL,
            birth_year     INT NOT NULL CHECK (birth_year BETWEEN 1900 AND 2025),
            birth_city_id  INT NOT NULL REFERENCES city(city_id),
            ceo_start_year INT NOT NULL CHECK (ceo_start_year BETWEEN 1900 AND 2100),
            CHECK (ceo_start_year > birth_year)
        );

        CREATE TABLE company (
            company_id    SERIAL PRIMARY KEY,
            company_name  VARCHAR(100) NOT NULL UNIQUE,
            founding_year INT NOT NULL CHECK (founding_year BETWEEN 1800 AND 2100),
            hq_city_id    INT NOT NULL REFERENCES city(city_id),
            ceo_id        INT NOT NULL UNIQUE REFERENCES ceo(ceo_id)
        );
    """)
    conn.commit()
    print("ceo and company created")
except Exception as e:
    conn.rollback()
    print("Error:", e)

ceo and company created


In [43]:
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' ORDER BY table_name;
""")
print(cur.fetchall())
conn.commit()

[('ceo',), ('city',), ('company',), ('parts',)]


In [44]:
cities = [
    # HQ cities
    ('Cupertino',     'California',          'USA'),
    ('Armonk',        'New York',            'USA'),
    ('Seattle',       'Washington',          'USA'),
    ('Detroit',       'Michigan',            'USA'),
    ('Tokyo',         None,                  'Japan'),
    ('Hangzhou',      'Zhejiang',            'China'),
    ('Paris',         None,                  'France'),
    ('Suwon',         'Gyeonggi-do',         'South Korea'),
    ('Bentonville',   'Arkansas',            'USA'),
    # Birthplaces
    ('Mobile',        'Alabama',             'USA'),
    ('West Godavari', 'Andhra Pradesh',      'India'),
    ('Scarsdale',     'New York',            'USA'),
    ('Royal Oak',     'Michigan',            'USA'),
    ('Yamaguchi',     'Yamaguchi Prefecture','Japan'),
    ('Unknown',       'Zhejiang',            'China'),
    ('Milan',         'Lombardy',            'Italy'),
    ('Seoul',         None,                  'South Korea'),
    ('Jacksonville',  'Arkansas',            'USA'),
]

try:
    cur.executemany(
        "INSERT INTO city (city_name, state_name, country_name) VALUES (%s, %s, %s)",
        cities
    )
    conn.commit()
    print(f"Inserted {len(cities)} cities")
except Exception as e:
    conn.rollback()
    print("Error:", e)
    

Inserted 18 cities


In [49]:
ceos = [
    ('Tim', 'Cook', 1960, 2011, 'Mobile', 'Alabama', 'USA'),
    ('Arvind', 'Krishna', 1962, 2020, 'West Godavari', 'Andhra Pradesh', 'India'),
    ('Andy', 'Jassy', 1968, 2021, 'Scarsdale', 'New York', 'USA'),
    ('Mary', 'Barra', 1961, 2014, 'Royal Oak', 'Michigan', 'USA'),
    ('Hiroki','Totoki', 1964, 2025, 'Yamaguchi', 'Yamaguchi Prefecture', 'Japan'),
    ('Eddie', 'Wu', 1975, 2023, 'Unknown', 'Zhejiang', 'China'),
    ('Luca', 'de Meo', 1967, 2025, 'Milan', 'Lombardy', 'Italy'),
    ('Jun', 'Young-hyun', 1960, 2025, 'Seoul', None, 'South Korea'),
    ('John', 'Furner', 1974, 2026, 'Jacksonville', 'Arkansas', 'USA'),
]

try:
    cur.executemany("""
        INSERT INTO ceo (first_name, last_name, birth_year, ceo_start_year, birth_city_id)
        SELECT %s, %s, %s, %s, city_id
        FROM city
        WHERE city_name = %s
          AND state_name IS NOT DISTINCT FROM %s
          AND country_name = %s
    """, ceos)

    if cur.rowcount != len(ceos):
        raise ValueError(f"Expected {len(ceos)} CEOs, inserted {cur.rowcount}")

    conn.commit()
    print(f"Inserted {len(ceos)} CEOs")
except Exception as e:
    conn.rollback()
    print("Error:", e)

Inserted 9 CEOs


In [50]:
companies = [
    # (company, founding_year, hq_city, hq_state, hq_country, ceo_last_name)
    ('Apple, Inc.', 1976, 'Cupertino', 'California', 'USA', 'Cook'),
    ('IBM', 1911, 'Armonk', 'New York', 'USA', 'Krishna'),
    ('Amazon', 1994, 'Seattle', 'Washington',  'USA', 'Jassy'),
    ('General Motors', 1908, 'Detroit', 'Michigan', 'USA', 'Barra'),
    ('Sony Group Corporation',1946, 'Tokyo', None, 'Japan', 'Totoki'),
    ('Alibaba Group', 1999, 'Hangzhou', 'Zhejiang', 'China', 'Wu'),
    ('Kering SA', 1963, 'Paris', None, 'France', 'de Meo'),
    ('Samsung Electronics', 1969, 'Suwon', 'Gyeonggi-do', 'South Korea', 'Young-hyun'),
    ('Walmart', 1962, 'Bentonville', 'Arkansas', 'USA', 'Furner'),
]

try:
    cur.executemany("""
        INSERT INTO company (company_name, founding_year, hq_city_id, ceo_id)
        SELECT %s, %s, ci.city_id, ce.ceo_id
        FROM city ci, ceo ce
        WHERE ci.city_name = %s
          AND ci.state_name IS NOT DISTINCT FROM %s
          AND ci.country_name = %s
          AND ce.last_name = %s
    """, companies)
    if cur.rowcount != len(companies):
        raise ValueError(f"Expected {len(companies)} companies, inserted {cur.rowcount}")
        
    conn.commit()
    print(f"Inserted {len(companies)} companies")
except Exception as e:
    conn.rollback()
    print("Error:", e)

Inserted 9 companies


In [51]:
for t in ['city', 'ceo', 'company']:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(t, cur.fetchone()[0])
conn.commit()

city 18
ceo 9
company 9


In [52]:
tests = {
    "Foreign key (company points to nonexistent CEO)":
        "INSERT INTO company (company_name, founding_year, hq_city_id, ceo_id) VALUES ('FakeCo', 2000, 1, 999)",
    "Unique (duplicate company name)":
        "INSERT INTO company (company_name, founding_year, hq_city_id, ceo_id) VALUES ('IBM', 1911, 1, 1)",
    "Check (CEO started before being born)":
        "INSERT INTO ceo (first_name, last_name, birth_year, birth_city_id, ceo_start_year) VALUES ('A', 'B', 1980, 1, 1970)",
    "Not null (missing last name)":
        "INSERT INTO ceo (first_name, birth_year, birth_city_id, ceo_start_year) VALUES ('A', 1980, 1, 2010)",
    "Unique index (duplicate city with no state)":
        "INSERT INTO city (city_name, state_name, country_name) VALUES ('Tokyo', NULL, 'Japan')",
}

for name, sql in tests.items():
    try:
        cur.execute(sql)
        conn.rollback()      # undo even if it unexpectedly succeeds
        print(f"NOT BLOCKED: {name}")
    except Exception as e:
        conn.rollback()
        print(f"Blocked -> {name}\n    {type(e).__name__}")

Blocked -> Foreign key (company points to nonexistent CEO)
    ForeignKeyViolation
Blocked -> Unique (duplicate company name)
    UniqueViolation
Blocked -> Check (CEO started before being born)
    CheckViolation
Blocked -> Not null (missing last name)
    NotNullViolation
Blocked -> Unique index (duplicate city with no state)
    UniqueViolation


In [57]:
conn.rollback()

In [58]:
df = run("""
SELECT co.company_name, co.founding_year,
       hq.city_name AS hq_city, hq.country_name AS hq_country,
       ce.first_name, ce.last_name, ce.birth_year, ce.ceo_start_year,
       bc.state_name AS birth_state, bc.country_name AS birth_country
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city hq ON co.hq_city_id = hq.city_id
JOIN city bc ON ce.birth_city_id = bc.city_id
ORDER BY co.company_id;
""")

# Q10 check
print(df[df.ceo_start_year > 2014].company_name.tolist())

# Q8 check
print(df[2026 - df.birth_year < 60][['company_name', 'last_name']])

['IBM', 'Amazon', 'Sony Group Corporation', 'Alibaba Group', 'Kering SA', 'Samsung Electronics', 'Walmart']
    company_name last_name
2         Amazon     Jassy
5  Alibaba Group        Wu
6      Kering SA    de Meo
8        Walmart    Furner


In [59]:
def run(sql):
    try:
        cur.execute(sql)
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
        conn.commit()
        return pd.DataFrame(rows, columns=cols)
    except Exception:
        conn.rollback()
        raise

In [60]:
# Q1. What are the first names of the CEOs of the companies headquartered in the USA?
run("""
SELECT ce.first_name
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city ci ON co.hq_city_id = ci.city_id
WHERE ci.country_name = 'USA';
""")

,first_name
0,Tim
1,Arvind
2,Andy
3,Mary
4,John


In [61]:
# Q2. Which city hosts the headquarters of the company whose CEO has a last name Totoki?
run("""
SELECT ci.city_name
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city ci ON co.hq_city_id = ci.city_id
WHERE ce.last_name = 'Totoki';
""")

,city_name
0,Tokyo


In [62]:
# Q3. What are the founding years of the companies whose CEOs were born between 1960 and 1969?
run("""
SELECT co.company_name, co.founding_year
FROM company co
JOIN ceo ce ON co.ceo_id = ce.ceo_id
WHERE ce.birth_year BETWEEN 1960 AND 1969;
""")

,company_name,founding_year
0,"Apple, Inc.",1976
1,IBM,1911
2,Amazon,1994
3,General Motors,1908
4,Sony Group Corporation,1946
5,Kering SA,1963
6,Samsung Electronics,1969


In [63]:
# Q4. How long has the CEO of the company founded in 1908 been serving in their position?
run(f"""
SELECT co.company_name, ce.first_name, ce.last_name,
       {THIS_YEAR} - ce.ceo_start_year AS years_as_ceo
FROM company co
JOIN ceo ce ON co.ceo_id = ce.ceo_id
WHERE co.founding_year = 1908;
""")

,company_name,first_name,last_name,years_as_ceo
0,General Motors,Mary,Barra,12


In [64]:
# Q5. What is the age of the CEO of the company based in Hangzhou? 
run(f"""
SELECT ce.first_name, ce.last_name, {THIS_YEAR} - ce.birth_year AS age
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city ci ON co.hq_city_id = ci.city_id
WHERE ci.city_name = 'Hangzhou';
""")

,first_name,last_name,age
0,Eddie,Wu,51


In [66]:
# Q6. What’s the age of the company whose CEO was born in the state of Tennessee?
run(f"""
SELECT co.company_name, {THIS_YEAR} - co.founding_year AS company_age
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city bc ON ce.birth_city_id = bc.city_id
WHERE bc.state_name = 'Tennessee';
""")

,company_name,company_age


In [67]:
# Q7. What are the full names of CEOs of companies founded in the USA before 1965?
run("""
SELECT ce.first_name || ' ' || ce.last_name AS full_name,
       co.company_name, co.founding_year
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city ci ON co.hq_city_id = ci.city_id
WHERE ci.country_name = 'USA' AND co.founding_year < 1965;
""")

,full_name,company_name,founding_year
0,Arvind Krishna,IBM,1911
1,Mary Barra,General Motors,1908
2,John Furner,Walmart,1962


In [68]:
# Q8. Which companies have CEOs who are younger than 60?
run(f"""
SELECT co.company_name, ce.first_name, ce.last_name,
       {THIS_YEAR} - ce.birth_year AS age
FROM company co
JOIN ceo ce ON co.ceo_id = ce.ceo_id
WHERE {THIS_YEAR} - ce.birth_year < 60;
""")

,company_name,first_name,last_name,age
0,Amazon,Andy,Jassy,58
1,Alibaba Group,Eddie,Wu,51
2,Kering SA,Luca,de Meo,59
3,Walmart,John,Furner,52


In [69]:
# Q9. What is the headquarter city of the company whose CEO was born in India? 
run("""
SELECT co.company_name, hq.city_name
FROM company co
JOIN ceo  ce ON co.ceo_id = ce.ceo_id
JOIN city bc ON ce.birth_city_id = bc.city_id
JOIN city hq ON co.hq_city_id = hq.city_id
WHERE bc.country_name = 'India';
""")

,company_name,city_name
0,IBM,Armonk


In [70]:
# Q10. Which companies have CEOs who started in their positions after 2014? 
run("""
SELECT co.company_name, ce.first_name, ce.last_name, ce.ceo_start_year
FROM company co
JOIN ceo ce ON co.ceo_id = ce.ceo_id
WHERE ce.ceo_start_year > 2014;
""")

,company_name,first_name,last_name,ceo_start_year
0,IBM,Arvind,Krishna,2020
1,Amazon,Andy,Jassy,2021
2,Sony Group Corporation,Hiroki,Totoki,2025
3,Alibaba Group,Eddie,Wu,2023
4,Kering SA,Luca,de Meo,2025
5,Samsung Electronics,Jun,Young-hyun,2025
6,Walmart,John,Furner,2026
